# Subgraphs  子图
子图是指用作另一个图的节点的图 。

子图的用途包括：
- 构建多智能体系统
- 在多个图中重用一组节点
- 分布式开发：当您希望不同的团队独立地处理图中的不同部分时，您可以将每个部分定义为一个子图。只要遵循子图的接口（输入和输出模式），就可以在不了解子图任何细节的情况下构建父图。

## Define subgraph communication 定义子图通信
添加子图时，需要定义父图和子图如何通信：
| 模式 | 何时使用 | 状态模式 |
|------|----------|-----------|
| **在节点内调用子图** | 父图和子图的状态模式不同（没有共享键），或需要在两者之间进行状态转换 | 编写一个包装函数，将父状态映射为子图输入，并将子图输出再映射回父状态 |
| **将子图作为节点添加** | 父图和子图共享状态键 —— 子图从与父图相同的通道读取和写入数据 | 直接将编译后的子图传递给 `add_node` —— 无需包装函数 |

### Call a subgraph inside a node 调用节点内的子图
当父图和子图具有不同的状态模式 （没有共享键）时，应在节点函数内部调用子图。这在多智能体系统中，当需要为每个智能体保留私有消息历史记录时很常见。

节点函数在调用子图之前将父状态转换为子图状态，并在返回之前将结果转换回父状态。

In [ ]:
# Grandchild graph
from typing_extensions import TypedDict
from langgraph.graph.state import StateGraph, START, END

class GrandChildState(TypedDict):
    my_grandchild_key: str

def grandchild_1(state: GrandChildState) -> GrandChildState:
    # NOTE: child or parent keys will not be accessible here
    return {"my_grandchild_key": state["my_grandchild_key"] + ", how are you"}


grandchild = StateGraph(GrandChildState)
grandchild.add_node("grandchild_1", grandchild_1)

grandchild.add_edge(START, "grandchild_1")
grandchild.add_edge("grandchild_1", END)

grandchild_graph = grandchild.compile()

# Child graph
class ChildState(TypedDict):
    my_child_key: str

def call_grandchild_graph(state: ChildState) -> ChildState:
    # NOTE: parent or grandchild keys won't be accessible here
    grandchild_graph_input = {"my_grandchild_key": state["my_child_key"]}
    grandchild_graph_output = grandchild_graph.invoke(grandchild_graph_input)
    return {"my_child_key": grandchild_graph_output["my_grandchild_key"] + " today?"}

child = StateGraph(ChildState)
# We're passing a function here instead of just compiled graph (`grandchild_graph`)
child.add_node("child_1", call_grandchild_graph)
child.add_edge(START, "child_1")
child.add_edge("child_1", END)
child_graph = child.compile()

# Parent graph
class ParentState(TypedDict):
    my_key: str

def parent_1(state: ParentState) -> ParentState:
    # NOTE: child or grandchild keys won't be accessible here
    return {"my_key": "hi " + state["my_key"]}

def parent_2(state: ParentState) -> ParentState:
    return {"my_key": state["my_key"] + " bye!"}

def call_child_graph(state: ParentState) -> ParentState:
    child_graph_input = {"my_child_key": state["my_key"]}
    child_graph_output = child_graph.invoke(child_graph_input)
    return {"my_key": child_graph_output["my_child_key"]}

parent = StateGraph(ParentState)
parent.add_node("parent_1", parent_1)
# We're passing a function here instead of just a compiled graph (`child_graph`)
parent.add_node("child", call_child_graph)
parent.add_node("parent_2", parent_2)

parent.add_edge(START, "parent_1")
parent.add_edge("parent_1", "child")
parent.add_edge("child", "parent_2")
parent.add_edge("parent_2", END)

parent_graph = parent.compile()

for chunk in parent_graph.stream({"my_key": "Bob"}, subgraphs=True, version="v2"):
    if chunk["type"] == "updates":
        print(chunk["ns"], chunk["data"])

### Add a subgraph as a node 添加子图作为节点
当父图和子图共享状态键时，可以直接将编译好的子图传递给 add_node 。无需任何包装函数——子图会自动从父图的状态通道读取和写入数据。例如，在多智能体系统中，智能体通常通过共享的消息键进行通信。
1. 定义子图工作流（如下例中的 subgraph_builder ）并编译它。
2. 在定义父图工作流时，将编译后的子图传递给 add_node 方法。

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph.state import StateGraph, START

# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # shared with parent graph state
    bar: str  # private to SubgraphState

def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}

def subgraph_node_2(state: SubgraphState):
    # note that this node is using a state key ('bar') that is only available in the subgraph
    # and is sending update on the shared state key ('foo')
    return {"foo": state["foo"] + state["bar"]}

subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()

# Define parent graph
class ParentState(TypedDict):
    foo: str

def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}

builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream({"foo": "foo"}, version="v2"):
    if chunk["type"] == "updates":
        print(chunk["data"])

## Subgraph persistence  子图持久性
.compile() 函数中的 checkpointer 参数控制子图的持久性：

| 模式 | checkpointer= | 行为 |
|------|---------------|------|
| **每次调用（Per-invocation）** | `None`（默认） | 每次调用都会重新开始，并继承父级的 checkpointer，以支持单次调用内的中断与持久执行 |
| **每线程（Per-thread）** | `True` | 同一线程上的调用会累积状态，每次调用都会从上一次中断的位置继续执行 |
| **无状态（Stateless）** | `False` | 不使用 checkpoint，行为类似普通函数调用，不支持中断或持久执行 |

对于大多数应用场景，包括子智能体处理独立请求的多智能体系统，按调用次数进行分配是正确的选择。当子智能体需要多轮对话记忆时（例如，研究助手需要通过多次交流来构建上下文），则应使用按线程进行分配。

### Stateful  有状态的
有状态子图继承父图的检查指针，从而实现中断 、 持久执行和状态检查。两种有状态模式的区别在于状态保留的时间长短。
#### Per-invocation (default)  每次调用（默认）
> 对于大多数应用程序，包括将子代理作为工具调用的多代理系统，这是推荐模式。它支持中断、 持久执行和并行调用，同时保持每次调用的隔离性。

当每次对子图的调用都是独立的，并且子代理不需要记住之前调用的任何信息时，可以使用按调用持久化。这是最常见的模式，尤其适用于多代理系统，其中子代理处理诸如“查找此客户的订单”或“汇总此文档”之类的一次性请求。

省略 checkpointer 或将其设置为 None 。每次调用都会重新开始，但在单次调用中，子图会继承父图的检查点，并且可以使用 interrupt() 来暂停和恢复。

每次调用都可以使用 interrupt() 来暂停和恢复。将 interrupt() 添加到工具函数中，以便在继续执行之前需要用户批准：

In [ ]:
@tool# type: ignore
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    interrupt("continue?")# type: ignore
    return f"Info about {fruit_name}"

config = {"configurable": {"thread_id": "1"}}

# Invoke - the subagent's tool calls interrupt()
response = agent.invoke(# type: ignore
    {"messages": [{"role": "user", "content": "Tell me about apples"}]},
    config=config,
)
# response contains __interrupt__

# Resume - approve the interrupt
response = agent.invoke(Command(resume=True), config=config)# type: ignore
# Subagent message count: 4

每次调用都会从一个新的子代理状态开始。子代理不会记住之前的调用：

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# First call
response = agent.invoke(# type: ignore
    {"messages": [{"role": "user", "content": "Tell me about apples"}]},
    config=config,
)
# Subagent message count: 4

# Second call - subagent starts fresh, no memory of apples
response = agent.invoke(# type: ignore
    {"messages": [{"role": "user", "content": "Now tell me about bananas"}]},
    config=config,
)
# Subagent message count: 4 (still fresh!)

多次调用同一个子图不会发生冲突，因为每次调用都会获得自己的检查点命名空间：

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# LLM calls ask_fruit_expert for both apples and bananas
response = agent.invoke(# type: ignore
    {"messages": [{"role": "user", "content": "Tell me about apples and bananas"}]},
    config=config,
)
# Subagent message count: 4 (apples - fresh)
# Subagent message count: 4 (bananas - fresh)

#### Per-thread  每个线程
当子代理需要记住之前的交互时，可以使用线程级持久化。例如，研究助手需要通过多次交流来积累上下文信息，或者编码助手需要跟踪已编辑的文件。子代理的对话历史记录和数据会在同一线程上的每次调用中累积。每次调用都会从上次中断的地方继续。

使用 checkpointer=True 启用此行为。

> 每个线程的子图不支持并行工具调用。当 LLM 将每个线程的子代理作为工具访问时，它可能会尝试并行多次调用该工具（例如，同时向水果专家询问苹果和香蕉）。这会导致检查点冲突，因为两次调用都写入了同一个命名空间。
> 使用 LangChain 的 ToolCallLimitMiddleware 来防止这种情况。如果您使用的是纯 LangGraph StateGraph ，则需要自行防止并行工具调用——例如，通过配置模型来禁用并行工具调用，或者添加逻辑来确保不会多次并行调用同一个子图。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

@tool
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    return f"Info about {fruit_name}"

# Subagent with checkpointer=True for persistent state
fruit_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[fruit_info],
    prompt="You are a fruit expert. Use the fruit_info tool. Respond in one sentence.",
    checkpointer=True,
)

# Wrap subagent as a tool for the outer agent
@tool
def ask_fruit_expert(question: str) -> str:
    """Ask the fruit expert. Use for ALL fruit questions."""
    response = fruit_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content

# Outer agent with checkpointer
# Use ToolCallLimitMiddleware to prevent parallel calls to per-thread subagents,
# which would cause checkpoint conflicts.
agent = create_agent(
    model="gpt-4.1-mini",
    tools=[ask_fruit_expert],
    prompt="You have a fruit expert. ALWAYS delegate fruit questions to ask_fruit_expert.",
    middleware=[
        ToolCallLimitMiddleware(tool_name="ask_fruit_expert", run_limit=1),
    ],
    checkpointer=MemorySaver(),
)

每个线程的子代理都支持 interrupt() 就像每个调用都支持 interrupt() 函数一样。向工具函数添加 interrupt() 函数以要求用户批准：

In [ ]:
@tool
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    interrupt("continue?")
    return f"Info about {fruit_name}"

config = {"configurable": {"thread_id": "1"}}

# Invoke - the subagent's tool calls interrupt()
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about apples"}]},
    config=config,
)
# response contains __interrupt__

# Resume - approve the interrupt
response = agent.invoke(Command(resume=True), config=config)
# Subagent message count: 4

状态会在多次调用中累积——子代理会记住过去的对话：

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# First call
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about apples"}]},
    config=config,
)
# Subagent message count: 4

# Second call - subagent REMEMBERS apples conversation
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Now tell me about bananas"}]},
    config=config,
)
# Subagent message count: 8 (accumulated!)

当您有多个不同的线程子图（例如，水果专家和蔬菜专家）时，每个子图都需要自己的存储空间，以避免检查点相互覆盖。这称为命名空间隔离 。

如果在节点内部调用子图 ，LangGraph 会根据调用顺序（第一次调用、第二次调用等）分配命名空间。这意味着重新排序调用可能会导致子图加载的状态不一致。为了避免这种情况，请将每个子代理包装在具有唯一节点名称的独立 StateGraph 中——这样每个子图就拥有一个稳定且唯一的命名空间：

In [ ]:
from langgraph.graph import MessagesState, StateGraph

def create_sub_agent(model, *, name, **kwargs):
    """Wrap an agent with a unique node name for namespace isolation."""
    agent = create_agent(model=model, name=name, **kwargs)
    return (
        StateGraph(MessagesState)
        .add_node(name, agent)  # unique name → stable namespace  #
        .add_edge("__start__", name)
        .compile()
    )

fruit_agent = create_sub_agent(
    "gpt-4.1-mini", name="fruit_agent",
    tools=[fruit_info], prompt="...", checkpointer=True,
)
veggie_agent = create_sub_agent(
    "gpt-4.1-mini", name="veggie_agent",
    tools=[veggie_info], prompt="...", checkpointer=True,# type: ignore
)

config = {"configurable": {"thread_id": "1"}}

# First call - LLM calls both fruit and veggie experts
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about cherries and broccoli"}]},
    config=config,
)
# Fruit subagent message count: 4
# Veggie subagent message count: 4

# Second call - both agents accumulate independently
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Now tell me about oranges and carrots"}]},
    config=config,
)
# Fruit subagent message count: 8 (remembers cherries!)
# Veggie subagent message count: 8 (remembers broccoli!)

作为节点添加的子图已经自动获得基于名称的命名空间，因此不需要此包装器。


### Stateless  无国籍人士
当您希望像普通函数调用一样运行子代理，且不涉及检查点开销时，请使用此选项。子图无法暂停/恢复，也无法从持久执行中受益。编译时请使用 checkpointer=False 。
### Checkpointer reference  检查点引用
使用 .compile() 中的 checkpointer 参数控制子图持久性：

In [ ]:
subgraph = builder.compile(checkpointer=False)  # or True / None

| 特征 | Per-invocation（默认） | Per-thread | Stateless |
|------|------------------------|------------|-----------|
| `checkpointer=` | `None` | `True` | `False` |
| 中断（HITL） | ✅ | ✅ | ❌ |
| 多轮记忆 | ❌ | ✅ | ❌ |
| 多次调用（不同子图） | ✅ | ⚠️ | ✅ |
| 多次调用（同一子图） | ✅ | ❌ | ✅ |
| 状态检查 | ⚠️ | ✅ | ❌ |

- 中断（HITL） ：子图可以使用 interrupt() 暂停执行并等待用户输入，然后从中断的地方恢复执行。
- 多轮内存 ：子图在同一线程内的多次调用中保持其状态。每次调用都会从上次中断的地方继续，而不是从头开始。
- 多次调用（不同的子图） ：可以在单个节点内调用多个不同的子图实例，而不会发生检查点命名空间冲突。
- 多次调用（同一子图） ：同一个子图实例可以在单个节点内被多次调用。使用有状态持久化时，这些调用会写入同一个检查点命名空间，从而导致冲突——请改用每次调用独立的持久化。
- 状态检查 ：子图的状态可通过 get_state(config, subgraphs=True) 获取，用于调试和监控。

## View subgraph state  查看子图状态
启用持久化后，可以使用 `subgraphs` 选项检查子图状态。如果使用无状态检查点（ checkpointer=False ），则不会保存子图检查点，因此无法查看子图状态。
> 查看子图状态的前提是 LangGraph 能够静态地发现该子图——也就是说，它必须作为节点添加或在另一个节点内部被调用 。如果子图在工具函数或其他间接调用方式（例如， 子代理模式）中被调用，则此方法无效。无论嵌套如何，中断仍然会传播到顶层图。

仅返回当前调用时的子图状态。每次调用都从头开始。

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class State(TypedDict):
    foo: str

# Subgraph
def subgraph_node_1(state: State):
    value = interrupt("Provide value:")
    return {"foo": state["foo"] + value}

subgraph_builder = StateGraph(State)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph = subgraph_builder.compile()  # inherits parent checkpointer

# Parent graph
builder = StateGraph(State)
builder.add_node("node_1", subgraph)
builder.add_edge(START, "node_1")

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "1"}}

graph.invoke({"foo": ""}, config)

# View subgraph state for the current invocation
subgraph_state = graph.get_state(config, subgraphs=True).tasks[0].state  

# Resume the subgraph
graph.invoke(Command(resume="bar"), config)

返回此线程上所有调用中累积的子图状态。

In [ ]:
from langgraph.graph import START, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver

# Subgraph with its own persistent state
subgraph_builder = StateGraph(MessagesState)
# ... add nodes and edges
subgraph = subgraph_builder.compile(checkpointer=True)

# Parent graph
builder = StateGraph(MessagesState)
builder.add_node("agent", subgraph)
builder.add_edge(START, "agent")

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "1"}}

graph.invoke({"messages": [{"role": "user", "content": "hi"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "what did I say?"}]}, config)

# View accumulated subgraph state (includes messages from both invocations)
subgraph_state = graph.get_state(config, subgraphs=True).tasks[0].state

## Stream subgraph outputs  流子图输出
要将子图的输出包含在流式输出中，可以在父图的 stream 方法中设置 subgraphs 选项。这将同时流式传输父图及其所有子图的输出。

In [ ]:
for chunk in graph.stream(
    {"foo": "foo"},
    subgraphs=True,
    stream_mode="updates",
    version="v2",
):
    print(chunk["type"])  # "updates"
    print(chunk["ns"])    # () for root, ("node_2:<task_id>",) for subgraph
    print(chunk["data"])  # {"node_name": {"key": "value"}}